# Importação de Pacotes

In [13]:
#leitura da base de dados
import pandas as pd
from pathlib import Path
import parquet

#modelo preditivo escolhido
import catboost as cb
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

#validação cruzada
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, HalvingGridSearchCV
import numpy as np

#vetorização
from langchain_experimental.text_splitter import SemanticChunker 
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

#métricas
import matplotlib
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, classification_report

#pipelines
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
def estimadores(modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)

    acuracia = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    roc_auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class='ovr',
        average='weighted'
    )

    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test)

    print(f"""
        Acurácia: {acuracia:.3f}
        Recall (weighted): {recall:.3f}
        F1-score (weighted): {f1:.3f}
        ROC AUC (ovr): {roc_auc:.3f}
        """)

    print(classification_report(y_test, y_pred))

## Leitura DataFrame

In [3]:
direcao = Path("../..") / "data"
caminho = direcao / "erro_medico_tidy_final.parquet"


df_erro_simples = pd.read_parquet(caminho)

df = df_erro_simples

### Escolha do Modelo

In [4]:
modelo= CatBoostClassifier(auto_class_weights='Balanced')

### Escolha dos HiperParâmetros

In [5]:
parametros = {
    'modelo__iterations': [300],
    'modelo__depth': [6],
    'modelo__learning_rate': [0.1],
    'modelo__verbose': [0]
}

In [6]:
df.head(2)

,n_processo,escopo,data_de_disponibilizacao,decisao,morais_ped,morais_rec,materiais_ped,materiais_rec,classe,assunto,...,tem_hospital,tem_plano_saude,tem_ente_publico,tem_medico_individual,n_adv_autor,n_adv_reu,tem_perito,tem_denuncia_lide,tem_assistente,resumo_caso
0,10169222220248260564,sim,2026-02-24,proced,100000.0,55000.0,-1.0,-1.0,Procedimento Comum Cível,Serviços de Saúde,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,## Identificação do Caso\n- Número do processo...
1,10003518420258260355,sim,2026-02-23,improced,151800.0,0.0,75900.0,0.0,Procedimento Comum Cível,Serviços de Saúde,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,## Identificação do Caso\n- **Número do proces...


In [7]:
df.columns

Index(['n_processo', 'escopo', 'data_de_disponibilizacao', 'decisao',
       'morais_ped', 'morais_rec', 'materiais_ped', 'materiais_rec', 'classe',
       'assunto', 'foro', 'vara', 'juiz', 'data_distribuicao', 'valor_acao',
       'area', 'n_autores', 'n_reus', 'tem_hospital', 'tem_plano_saude',
       'tem_ente_publico', 'tem_medico_individual', 'n_adv_autor', 'n_adv_reu',
       'tem_perito', 'tem_denuncia_lide', 'tem_assistente', 'resumo_caso'],
      dtype='object')

### Preparação Embedding

In [8]:
corpus = df["resumo_caso"].dropna().astype(str).tolist()

**versão sem tqdm**

model_name = "rufimelo/bert-large-portuguese-cased-sts"

model_kwargs = {"device" : "cpu"}

encode_kwargs = {"normalize_embeddings" : False}

embeddings = HuggingFaceEmbeddings(
    model_name = model_name,
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

vector_store = InMemoryVectorStore(embeddings)
text_splitter = SemanticChunker(embeddings, breakpoint_threshold_type="gradient")

texts = text_splitter.create_documents(corpus)
document_ids = vector_store.add_documents(documents=texts)

obj_embed = embeddings.embed_documents(corpus)

df_embed = pd.DataFrame(obj_embed)


In [ ]:
from tqdm.auto import tqdm
import pandas as pd

model_name = "rufimelo/bert-large-portuguese-cased-sts"

model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

vector_store = InMemoryVectorStore(embeddings)
text_splitter = SemanticChunker(embeddings, breakpoint_threshold_type="gradient")

# 1) Chunking com progresso
texts = []
for doc in tqdm(corpus, desc="Gerando chunks"):
    texts.extend(text_splitter.create_documents([doc]))

# 2) Indexação em lotes com progresso
document_ids = []
batch_size = 64

for i in tqdm(range(0, len(texts), batch_size), desc="Adicionando ao vector store"):
    batch = texts[i:i + batch_size]
    document_ids.extend(vector_store.add_documents(documents=batch))

# 3) Embeddings com progresso
obj_embed = []
batch_size = 8

for i in tqdm(range(0, len(corpus), batch_size), desc="Calculando embeddings"):
    batch = corpus[i:i + batch_size]
    obj_embed.extend(embeddings.embed_documents(batch))

df_embed = pd.DataFrame(obj_embed)

Invalid model-index. Not loading eval results into CardData.
Gerando chunks:   7%|▋         | 153/2048 [1:42:40<23:31:40, 44.70s/it]

In [ ]:
#axis=1 concatena colunas
df_vet = pd.concat([df, df_embed], axis= 1)

# Aplicação de Pipelines

### CBC

In [ ]:
modelo= CatBoostClassifier(auto_class_weights='Balanced')

parametros = {
    'modelo__iterations': [300],
    'modelo__depth': [4, 6, 8],
    'modelo__learning_rate': [0.05, 0.1],
    'modelo__verbose': [0]
}

In [ ]:
lista_filtro = ['n_processo', 'escopo', 'data_de_disponibilizacao', 'decisao',
                'morais_ped', 'morais_rec', 'materiais_ped', 'materiais_rec', 
                'classe', 'assunto', 'foro', 'vara', 'juiz', 'data_distribuicao', 
                'area']


X = df_vet.drop(columns=lista_filtro)

y = df_vet["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)


pipeline = Pipeline([
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='f1_macro',
    refit= True,
    cv=5
)

searchCV_pipeline.fit(X_train, y_train)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)

### GBC

In [ ]:
modelo= GradientBoostingClassifier()

parametros = {
    'modelo__n_estimators': [100],
    'modelo__max_depth': [2, 6],
    'modelo__max_leaf_nodes': [10],
    'modelo__learning_rate': [0.05, 0.2]
    }

In [ ]:
lista_filtro = ['n_processo', 'escopo', 'data_de_disponibilizacao', 'decisao',
                'morais_ped', 'morais_rec', 'materiais_ped', 'materiais_rec', 
                'classe', 'assunto', 'foro', 'vara', 'juiz', 'data_distribuicao', 
                'area']


X = df_vet.drop(columns=lista_filtro)

y = df_vet["decisao"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=22)

num_prep = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_prep, lista_X),
    ])


pipeline = Pipeline([
    ('preprocessor', preprocessor)
    ('modelo', modelo)
    ])

searchCV_pipeline = RandomizedSearchCV(
    pipeline,
    parametros,
    scoring='roc_auc_ovr',
    refit= True,
    cv=5,
    verbose= 2
)


sample_weight = compute_sample_weight('balanced', y_train)
searchCV_pipeline.fit(X_train, y_train)

In [ ]:
estimadores(searchCV_pipeline, X_test, y_test)